In [1]:
import pandas as pd
import numpy as np
import json
import time
import os
from openai import OpenAI
from dotenv import load_dotenv
import warnings
import joblib

warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("=" * 80)
print("Agent #1: CF Quality Interpreter")
print("=" * 80)

# ============================================================================
# 1. Configuration and Data Loading
# ============================================================================
FEATURE_MAPPING = {
    'FN1_5': 'Tangible Assets', 'FN3_10': 'Liquidation Value Ratio', 'FN1_4': 'Inventory',
    'FN2_10_1': 'Prior-Year Net Income', 'FN2_1': 'Revenue', 'FN2_7': 'Non-Operating Income',
    'FN1_13_1': 'Prior-Year Total Assets', 'FN1_11_1': 'Prior-Year Accounts Receivable',
    'FN3_6': 'Reserve Ratio', 'FN2_1_1': 'Prior-Year Revenue', 'FN1_13': 'Total Assets',
    'FN1_11': 'Accounts Receivable', 'FN1_19': 'Total Liabilities', 'FN1_16': 'Borrowings',
    'FN3_3': 'Debt Service Coverage Ratio', 'PERF_12M': 'Bankruptcy Flag'
}

# Load data
try:
    # Optimal CF file generated in Step 4
    df_filtered = pd.read_csv('cf_results_filtered.csv')

    # Full evaluation details for comparative explanation
    df_details = pd.read_csv('cf_evaluation_details.csv')

    selected_features = joblib.load('selected_features_final.pkl')

    print(f"Analysis targets (optimal CFs): {len(df_filtered)} records")
    print(f"Reference pool (all candidates): {len(df_details)} records")

except FileNotFoundError:
    raise FileNotFoundError("Required CSV/PKL files not found. Please run previous steps first.")

# ============================================================================
# 2. Agent Class Definition
# ============================================================================
class CFInterpreter:
    def __init__(self):
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model = os.getenv('LLM_MODEL', 'gpt-4o-mini')

    def generate_interpretation(self, best_row, all_candidates):
        """
        Generate an interpretation report for the optimal CF scenario.
        """
        company_id = best_row['ID']

        # 1. Build quantitative comparison data
        other_cfs = all_candidates[all_candidates['ID'] == company_id]
        avg_score = other_cfs['Quality_Score'].mean()
        best_score = best_row['Quality_Score']

        # Extract top variable changes
        changes = {}
        for feat in selected_features:
            delta = best_row[f'Change_{feat}']
            if abs(delta) > 0.001:
                english_name = FEATURE_MAPPING.get(feat, feat)
                changes[english_name] = {
                    'original': best_row[f'Original_{feat}'],
                    'target': best_row[f'CF_{feat}'],
                    'delta': delta
                }

        # Sort by magnitude of change, keep top 5
        sorted_changes = dict(
            sorted(changes.items(), key=lambda item: abs(item[1]['delta']), reverse=True)[:5]
        )

        # 2. Construct prompt
        system_prompt = """
        You are a corporate credit rating analyst.
        Your task is to explain why the AI model selected a particular
        'financial improvement scenario (Counterfactual)' and what it means in practice.
        """

        user_prompt = f"""
        [Company ID: {company_id}] — AI Financial Improvement Proposal

        # 1. Selection Rationale (Why this scenario?)
        This scenario was selected from {len(other_cfs)} candidates with the highest
        quality score ({best_score:.2f}, vs. average {avg_score:.2f}).
        - Realism: {best_row['Realism']:.2f} (alignment with solvent-firm distribution)
        - Proximity: {best_row['Proximity']:.4f} (minimal change required for large effect)
        - Robustness: {best_row['Robustness']:.2f} (probability of maintaining approval under market fluctuation)

        # 2. Key Proposed Changes (What to change?)
        The following variables require adjustment:
        {json.dumps(sorted_changes, ensure_ascii=False, indent=2)}

        # Task
        Based on the data above, generate an interpretation report in the following JSON format.

        {{
            "selection_reason": "One-sentence summary of why this scenario was selected, citing quality indicators (realism, efficiency, etc.)",
            "business_meaning": "Explanation of what the proposed numerical changes mean in practice (e.g., asset disposal, debt repayment). 2–3 sentences.",
            "feasibility_assessment": "Assessment of whether these changes are realistically achievable given wholesale industry characteristics (Pass / Warning)",
            "expected_outcome": "Expected positive outcomes if implemented (e.g., reduction in bankruptcy probability)"
        }}
        """

        # 3. API call
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.3
            )
            return json.loads(response.choices[0].message.content)

        except Exception as e:
            return {"error": str(e)}

# ============================================================================
# 3. Execution and Export
# ============================================================================
if __name__ == "__main__":
    agent = CFInterpreter()
    results = []

    print(f"\nGenerating interpretations for {len(df_filtered)} firms...")

    # To test on first 5 rows only, use: df_filtered.iloc[:5].iterrows()
    target_rows = df_filtered.iterrows()

    from tqdm import tqdm
    for idx, row in tqdm(target_rows, total=len(df_filtered)):
        interpretation = agent.generate_interpretation(row, df_details)

        result_entry = row.to_dict()
        result_entry.update(interpretation)
        results.append(result_entry)

    # Convert to DataFrame
    final_df = pd.DataFrame(results)

    # Save results
    output_filename = 'agent1_interpretation_results.csv'
    final_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

    print("\n" + "=" * 80)
    print("Interpretation generation complete")
    print("=" * 80)
    print(f"Output file: {output_filename}")

    # Sample output
    if not final_df.empty:
        sample = final_df.iloc[0]
        print(f"\n[ID: {sample['ID']} Interpretation Results]")
        print(f"1. Selection reason: {sample.get('selection_reason')}")
        print(f"2. Business meaning: {sample.get('business_meaning')}")
        print(f"3. Feasibility assessment: {sample.get('feasibility_assessment')}")

Agent #1: CF Quality Interpreter
Analysis targets (optimal CFs): 266 records
Reference pool (all candidates): 1061 records

Generating interpretations for 266 firms...


100%|████████████████████████████████████████████████████████████████████████████████| 266/266 [15:08<00:00,  3.41s/it]


Interpretation generation complete
Output file: agent1_interpretation_results.csv

[ID: 75.0 Interpretation Results]
1. Selection reason: This scenario was selected due to its high quality score of 0.91, indicating strong realism, minimal required changes for significant effects, and robust performance under market fluctuations.
2. Business meaning: The proposed changes suggest a significant reduction in revenue and various financial metrics, indicating a strategic shift towards downsizing operations or divesting non-core assets. This could involve selling off accounts receivable and reducing operational costs, ultimately aiming to streamline the business and improve liquidity.
3. Feasibility assessment: Warning
